# Tsay Feature Research

Project One v1 through **Stage 4 only**. Stages 1–3 froze the
contract, built T01–T09 without outcomes, and evaluated the fixed
Development feature-evidence family. Stage 4 fits only the frozen
D0–D6, O0–O5, and E0–E3 nested ladder on Development through
2023-12-31.

Validation labels/performance, MGC research rows, and every 2024+
outcome remain unopened. The Tsay PDF remains traceability-only.

## 1. Verified continuation and deterministic Stage 4 run

The runner first verifies the Stage 3 `STATUS: READY`, contract and
amendment hashes, all Stage 3 artifact hashes, and the Development-only
access manifest. It then executes the fixed nested folds on CPU.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists() and (candidate / "CLAUDE.md").exists():
            return candidate
    raise RuntimeError(f"Repository root not found from {start}")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["PROJECT_COMPUTE"] = "cpu"

from scripts.run_tsay_feature_research import ARTIFACT_ROOT, run_stage4
from src.statistical_research.tsay_artifacts import load_parquet

CHECKPOINT = run_stage4()
CHECKPOINT

{'stage': 4,
 'stage_name': 'development_nested_models_oof_economics_and_freeze',
 'status': 'READY',
 'integrity_reasons': [],
 'research_verdict': 'NO_PREDICTIVE_MODEL_AUTHORIZED',
 'contract_sha256': 'cc95d3254c9f271a60fd0fa0042a24c69328044748348802aea1ab2817dc6e0f',
 'amendment_sha256': 'ada63583e1fb79d625c47a1100d8bfedfc9c07d666dcab96bc5eada6f55dfc72',
 'stage3_hash_manifest_sha256': 'f21c13aae376d101bd35bd306f41930dd76321c571c8395ff046db3840b4c047',
 'stage4_hash_manifest_sha256': '311e5bd783550781ac72e576b8b7b9251e26018e9b2a54e9122e35625244259a',
 'access_manifest_sha256': 'e0653cb4c4b35a57ec22e76f517170416f22e2e88563715480181d9662f4e933',
 'deployment_state_sha256': 'd82702c4d006ea3fe11ab35d3ce80d1546a6d8e381a5e09a260ea4718ea98c66',
 'policy_state_sha256': '0b9d2c1ce4d624de96302cbfcd1a69bbecf716f4e827ef9ed5f99570c7b0c3cb',
 'predictive_model_authorization': False,
 'directional_model_authorized': False,
 'opportunity_only_authorized': False,
 'opportunity_model_authorized': Fal

In [2]:
def load_json(name: str):
    return json.loads((ARTIFACT_ROOT / name).read_text(encoding="utf-8"))


prior = load_json("stage3_continuation_verification_stage4.json")
assert prior["status"] == "VERIFIED"
assert prior["stage3_status"] == "READY"
prior

{'access_manifest_sha256': '3b865d8bf3eb91cf18e279441c70adb1b0dc4724feaa4e80744e24c11d76a52a',
 'amendment_sha256': 'ada63583e1fb79d625c47a1100d8bfedfc9c07d666dcab96bc5eada6f55dfc72',
 'artifact_hashes_verified': 16,
 'contract_sha256': 'cc95d3254c9f271a60fd0fa0042a24c69328044748348802aea1ab2817dc6e0f',
 'immutable_source_hashes_verified': 5,
 'stage3_hash_manifest_sha256': 'f21c13aae376d101bd35bd306f41930dd76321c571c8395ff046db3840b4c047',
 'stage3_status': 'READY',
 'status': 'VERIFIED'}

## 2. Development access boundary

Stage 4 reads only GC Development rows through 2023-12-31 and only
the prescribed feature inputs, 60-minute targets, expansion label,
availability flag, and exit timestamp required for label purging.

In [3]:
access = load_json("access_manifest_stage4.json")
assert access["stage"] == 4
assert access["materialization_cutoff_date_ny"] == "2023-12-31"
assert all(
    record["returned_maximum_date"] is None
    or record["returned_maximum_date"] <= "2023-12-31"
    for record in access["records"]
)
pd.DataFrame(access["records"])[[
    "source", "requested_columns", "returned_row_count",
    "returned_research_partition_set", "returned_maximum_date",
]]

,source,requested_columns,returned_row_count,returned_research_partition_set,returned_maximum_date
0,data/processed/statistical_research/tsay_featu...,"[observation_id, decision_timestamp_utc, trade...",306230,[Development],2023-12-29
1,data/processed/statistical_research/feature_ma...,"[observation_id, decision_timestamp_utc, trade...",306230,[Development],2023-12-29
2,data/processed/statistical_research/fes_projec...,"[observation_id, decision_timestamp_utc, trade...",306230,[Development],2023-12-29
3,data/processed/statistical_research/forward_la...,"[observation_id, decision_timestamp_utc, trade...",306230,[Development],2023-12-29


## 3. Nested OOF model evidence

Every candidate uses the frozen outer folds, purged inner folds,
fold-local transforms, one-standard-error tuning, matched anchor rows,
stationary-bootstrap inference, and family max-statistic correction.
Negative and non-evaluable findings remain in the ledgers.

In [4]:
continuous = load_parquet(
    ARTIFACT_ROOT / "development_continuous_model_evidence.parquet"
)
classifier = load_parquet(
    ARTIFACT_ROOT / "development_classifier_model_evidence.parquet"
)
selection = load_parquet(
    ARTIFACT_ROOT / "development_architecture_selection.parquet"
)
display(continuous)
display(classifier)
display(load_parquet(ARTIFACT_ROOT / "development_model_max_stat_audit.parquet"))
selection

,role,architecture,entry_session,matched_rows,matched_dates,matched_anchor_fraction,missing_prediction_count,mean_daily_ic,anchor_mean_daily_ic,mean_paired_delta_ic,...,calibration_intercept,calibration_slope,top10_positive_delta_share,positive_outer_fold_fraction,coefficient_median_pairwise_cosine,coefficient_sign_agreement,qualifying_coefficients,family_max_stat_adjusted_p,failed_gates,status
0,directional,D2,London,90541.0,504.0,1.000000,0.0,-0.097843,0.026503,-0.124346,...,-0.014233,-0.667351,0.159951,NaN,0.923532,0.941667,5.0,1.000000,mean_ic|delta|delta_ci|positive_folds|calibrat...,DEV_REJECTED
1,directional,D3,London,90541.0,504.0,1.000000,0.0,0.046083,0.026503,0.019581,...,-0.003791,-0.604770,0.127185,NaN,0.954077,0.898148,18.0,0.286857,delta_ci|positive_folds|calibration|family_max...,DEV_REJECTED
2,directional,D4,London,90006.0,504.0,0.994091,535.0,0.053293,0.026907,0.026386,...,0.063343,-0.009397,0.106749,NaN,0.896825,0.984375,8.0,0.224388,delta_ci|positive_folds|calibration|family_max...,DEV_REJECTED
3,directional,D5,London,90541.0,504.0,1.000000,0.0,0.084360,0.026503,0.057858,...,0.061298,0.028991,0.113686,NaN,0.856859,0.972418,71.0,0.034983,positive_folds|calibration|quantile_gates,DEV_REJECTED
4,directional,D6,London,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,model_not_fitted,NOT_AUTHORIZED
5,directional,D2,New York,150955.0,504.0,1.000000,0.0,0.120340,0.170248,-0.049908,...,0.016370,-1.253182,0.132699,NaN,0.669058,0.841667,5.0,1.000000,delta|delta_ci|positive_folds|calibration|fami...,DEV_REJECTED
6,directional,D3,New York,150955.0,504.0,1.000000,0.0,0.129491,0.170248,-0.040758,...,-0.017405,-0.376220,0.213338,NaN,0.982789,0.896930,19.0,1.000000,delta|delta_ci|positive_folds|calibration|fami...,DEV_REJECTED
7,directional,D4,New York,150449.0,504.0,0.996648,506.0,0.022241,0.171020,-0.148778,...,-0.064035,0.480847,0.167932,NaN,0.986322,0.981481,9.0,1.000000,delta|delta_ci|positive_folds|calibration|fami...,DEV_REJECTED
8,directional,D5,New York,150955.0,504.0,1.000000,0.0,0.180086,0.170248,0.009838,...,0.015244,-0.362179,0.117536,NaN,0.960909,0.980346,53.0,0.540230,delta|delta_ci|positive_folds|calibration|quan...,DEV_REJECTED
9,directional,D6,New York,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,model_not_fitted,NOT_AUTHORIZED


,architecture,entry_session,matched_rows,matched_dates,matched_anchor_fraction,missing_prediction_count,roc_auc,roc_auc_improvement,pr_auc,pr_auc_improvement,...,calibration_slope,ece,positive_prevalence,top10_positive_delta_share,coefficient_median_pairwise_cosine,coefficient_sign_agreement,qualifying_coefficients,family_max_stat_adjusted_p,failed_gates,status
0,E2,London,90541,504,1.0,0,0.624205,-0.003861,0.216145,0.004247,...,1.033933,0.010784,0.138867,0.174398,0.960461,0.986111,6,0.319340,brier|brier_ci|roc|family_max_stat,DEV_REJECTED
1,E3,London,90541,504,1.0,0,0.633442,0.005376,0.214905,0.003007,...,0.752720,0.017901,0.138867,0.146202,0.949167,0.944444,27,0.193403,brier|brier_ci|roc|calibration|family_max_stat,DEV_REJECTED
2,E2,New York,150955,504,1.0,0,0.680925,-0.110058,0.399600,-0.066669,...,0.898170,0.014669,0.233265,0.181509,0.983582,1.000000,9,1.000000,brier|brier_ci|roc|pr|log_loss|family_max_stat,DEV_REJECTED
3,E3,New York,150955,504,1.0,0,0.790739,-0.000244,0.472614,0.006344,...,0.890811,0.034972,0.233265,0.124682,0.988019,0.992647,17,0.001000,brier|roc,DEV_REJECTED


,family,entry_session,architecture,common_dates,original_standard_error,observed_t,adjusted_p_value,fixed_original_studentizer
0,directional,London,D2,504,0.017880,-6.954357,1.000000,True
1,directional,London,D3,504,0.011775,1.662868,0.286857,True
2,directional,London,D4,504,0.014109,1.870102,0.224388,True
3,directional,London,D5,504,0.018043,3.206655,0.034983,True
4,directional,New York,D2,504,0.016515,-3.021986,1.000000,True
5,directional,New York,D3,504,0.008031,-5.075112,1.000000,True
6,directional,New York,D4,504,0.014560,-10.218510,1.000000,True
7,directional,New York,D5,504,0.012886,0.763412,0.540230,True
8,opportunity,London,O2,504,0.012881,-1.689991,1.000000,True
9,opportunity,London,O3,504,0.005600,3.003464,0.008996,True


,role,selected_architecture,fully_passing_architectures,selection_status,shared_architecture_across_session_fits,family_common_dates,best_architecture,best_mean_two_session_improvement,best_bootstrap_standard_error,one_standard_error_threshold,selected_mean_two_session_improvement
0,directional,None,[],NO_MODEL_PASSED,True,504,None,NaN,NaN,NaN,NaN
1,opportunity,None,[],NO_MODEL_PASSED,True,504,None,NaN,NaN,NaN,NaN
2,expansion,None,[],NO_MODEL_PASSED,True,504,None,NaN,NaN,NaN,NaN


## 4. Specialized diagnostics and frozen policy state

Quantile, VAR, Kalman, calibration, convergence, and coefficient
stability diagnostics are persisted separately. Real bars and policy
economics remain unopened unless a directional architecture passes all
Development predictive gates.

In [5]:
display(load_parquet(ARTIFACT_ROOT / "development_quantile_diagnostics.parquet"))
display(load_parquet(ARTIFACT_ROOT / "development_classifier_reliability.parquet"))
display(load_parquet(ARTIFACT_ROOT / "development_var_diagnostics.parquet").head())
display(load_parquet(ARTIFACT_ROOT / "development_kalman_diagnostics.parquet").head())
policy = load_json("frozen_policy_state.json")
policy

,entry_session,q10_pinball_loss,q10_baseline_pinball_loss,q10_pinball_improvement,q10_empirical_below_probability,q50_pinball_loss,q50_baseline_pinball_loss,q50_pinball_improvement,q50_empirical_below_probability,q90_pinball_loss,...,q90_empirical_below_probability,matched_rows,matched_dates,central_80_coverage,central_80_absolute_error,mean_interval_width_atr,pre_repair_crossing_count,pre_repair_crossing_rate,post_repair_crossing_count,gate_pass
0,London,0.968965,0.965391,-0.003701,0.093600,2.017073,2.009086,-0.003976,0.495682,0.937883,...,0.890862,90541,504,0.797263,0.002737,12.621564,124,0.00137,0,False
1,New York,1.212168,1.260916,0.038661,0.106844,2.424295,2.418489,-0.002401,0.517180,1.232430,...,0.894667,150955,504,0.787824,0.012176,14.002006,80,0.00053,0,False


,architecture,entry_session,equal_count_bin,observation_count,date_balanced_weight,weighted_mean_probability,weighted_prevalence,absolute_gap
0,E2,London,1,9055,0.099864,0.077153,0.076051,0.001102
1,E2,London,2,9054,0.099914,0.097580,0.094106,0.003473
2,E2,London,3,9054,0.099898,0.110422,0.096218,0.014203
3,E2,London,4,9054,0.100005,0.121747,0.098149,0.023599
4,E2,London,5,9054,0.100060,0.133093,0.119038,0.014055
5,E2,London,6,9054,0.099931,0.145244,0.131950,0.013295
6,E2,London,7,9054,0.099919,0.159426,0.146564,0.012862
7,E2,London,8,9054,0.099827,0.177031,0.166923,0.010109
8,E2,London,9,9054,0.100485,0.202696,0.192614,0.010082
9,E2,London,10,9054,0.100098,0.271621,0.266566,0.005055


,diagnostic,component,lag,statistic,raw_p_value,holm_adjusted_p_value,holm_rejected_at_0_05,architecture,role,entry_session,outer_fold_id,maximum_companion_eigenvalue,calibration_intercept,calibration_slope,forecast_failure_count,fit_success
0,residual_ljung_box,return_1m_bps,5,3.623272,6.048225e-01,1.000000e+00,False,D4,directional,London,0,0.754877,-1.031027,2.535194,0,True
1,residual_ljung_box,return_1m_bps,10,7.116520,7.144008e-01,1.000000e+00,False,D4,directional,London,0,0.754877,-1.031027,2.535194,0,True
2,residual_ljung_box,return_1m_bps,20,17.159538,6.425912e-01,1.000000e+00,False,D4,directional,London,0,0.754877,-1.031027,2.535194,0,True
3,squared_residual_ljung_box,return_1m_bps,5,780.585117,1.833212e-166,3.299782e-165,True,D4,directional,London,0,0.754877,-1.031027,2.535194,0,True
4,squared_residual_ljung_box,return_1m_bps,10,992.802129,6.643287e-207,1.262225e-205,True,D4,directional,London,0,0.754877,-1.031027,2.535194,0,True


,diagnostic,lag,statistic,raw_p_value,holm_adjusted_p_value,holm_rejected_at_0_05,architecture,role,entry_session,outer_fold_id,q_value,r_value,log_q,log_r,innovation_nll,innovation_mean,innovation_std,fit_success
0,innovation_ljung_box,5,40.652996,1.102580e-07,4.410319e-07,True,O5,opportunity,London,0,0.008174,1.184370e-07,-4.806774,-15.948885,-21314.756688,-0.025019,1.001333,True
1,squared_innovation_arch_lm,5,19.604038,1.482568e-03,1.482568e-03,True,O5,opportunity,London,0,0.008174,1.184370e-07,-4.806774,-15.948885,-21314.756688,-0.025019,1.001333,True
2,innovation_ljung_box,10,47.284756,8.380603e-07,2.514181e-06,True,O5,opportunity,London,0,0.008174,1.184370e-07,-4.806774,-15.948885,-21314.756688,-0.025019,1.001333,True
3,squared_innovation_arch_lm,10,44.334347,2.866273e-06,5.732545e-06,True,O5,opportunity,London,0,0.008174,1.184370e-07,-4.806774,-15.948885,-21314.756688,-0.025019,1.001333,True
4,innovation_ljung_box,20,2977.272217,0.000000e+00,0.000000e+00,True,O5,opportunity,London,0,0.008174,1.184370e-07,-4.806774,-15.948885,-21314.756688,-0.025019,1.001333,True


{'development_economics_calculated': False,
 'policy_status': 'FROZEN_NO_POLICY',
 'real_bar_data_loaded': False,
 'reason': 'No fully passing Development directional architecture.',
 'validation_economics_may_open': False}

## 5. Persistence, hashes, and Stage 4 checkpoint

All Stage 4 artifacts were saved, reloaded, compared, and hashed before
this checkpoint. `STATUS` reports integrity separately from the research
verdict and authorization fields.

In [6]:
hashes = load_json("stage4_hash_manifest.json")
assert hashes["development_label_access_only"] is True
assert hashes["validation_label_access_absent"] is True
assert hashes["mgc_access_absent"] is True
assert hashes["post_2023_materialization_absent"] is True
assert CHECKPOINT["status"] == "READY"
assert CHECKPOINT["validation_outcomes_read"] is False
assert CHECKPOINT["mgc_research_rows_read"] is False
assert CHECKPOINT["post_2023_rows_materialized"] is False

print(f"STATUS: {CHECKPOINT['status']}")
print(f"RESEARCH_VERDICT: {CHECKPOINT['research_verdict']}")
print(
    "PREDICTIVE_MODEL_AUTHORIZATION: "
    f"{CHECKPOINT['predictive_model_authorization']}"
)
print(f"POLICY_AUTHORIZED: {CHECKPOINT['policy_authorized']}")
print(
    "VALIDATION_ECONOMICS_MAY_OPEN: "
    f"{CHECKPOINT['validation_economics_may_open']}"
)
print(f"STAGE_5_AUTHORIZED: {CHECKPOINT['stage5_authorized']}")

STATUS: READY
RESEARCH_VERDICT: NO_PREDICTIVE_MODEL_AUTHORIZED
PREDICTIVE_MODEL_AUTHORIZATION: False
POLICY_AUTHORIZED: False
VALIDATION_ECONOMICS_MAY_OPEN: False
STAGE_5_AUTHORIZED: False


Stage 4 stops here. Stage 5 has not begun and may run only after a
separate explicit continuation instruction and the recorded authorization.